In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
print(sys.executable)

X_df = pd.read_pickle('../data/processed/X_df.pkl')
y_df = pd.read_pickle('../data/processed/y_df.pkl')

from sklearn.model_selection import GroupShuffleSplit

groups = X_df['WT_name']

splitter = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)


train_idx, test_idx = next(splitter.split(X_df, y_df, groups=groups))

X_train, X_test  = X_df.iloc[train_idx].copy(), X_df.iloc[test_idx].copy()
y_train, y_test = y_df.iloc[train_idx].copy(), y_df.iloc[test_idx].copy()

/Users/shanewarland/miniforge3/envs/protein-ml/bin/python


In [12]:
from src.features import get_biochemical_features

X_train = get_biochemical_features(X_train)
X_test = get_biochemical_features(X_test)

## Grab numeric columns
feature_cols = ["position", "relative_position", "protein_length",
    "wt_hydrophobicity", "mut_hydrophobicity", "delta_hydrophobicity", 
    "wt_mw", "mut_mw","delta_mw", 
    "wt_charge", "mut_charge", "delta_charge", "wt_aromatic", "mut_aromatic"]

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

esm_model.eval()

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True, bias=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
    

In [5]:
sequence = X_train["aa_seq"].iloc[0]

tokens = tokenizer(
    sequence,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = esm_model(**tokens)

print(outputs.last_hidden_state.shape)

torch.Size([1, 49, 320])


This is the embedding of 1 amino acid seqeunce. The 49 is the length of the amino acid sequence + 2 special tokens. The final dimension 320 is the embedding dimension for this mode. 

In [8]:
with torch.no_grad():
    outputs = esm_model(**tokens)

residue_embeddings = outputs.last_hidden_state[:, 1:-1, :]

protein_embedding = residue_embeddings.mean(dim=1)

print(protein_embedding.shape)

torch.Size([1, 320])


A first test is to get the embeddings for all WT and for all mut protein sequence and input these into lasso like regression to determine if the protein seqeunce even in a linear model can outperform a random forest. 

In [ ]:
X_train.head()


5         QEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
6         EEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
7         NEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
8         HEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
9         REVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
                               ...                       
473755        SEEAEKRARELYKRGVSKEQIKEVLKRLGVDPKEVEEIVRRCK
473761        SEEAEKRARELYKRGVSKEQIKEVLKRLGVDPKEVEEIVRRIN
473764        SEEAEKRARELYKRGVSKEQIKEVLKRLGVDPKEVEEIVRRIR
473776        SEEAEKRARELYKRGVSKEQIKEVLKRLGVDPKEVEEIVRRIP
473777        SEEAEKRARELYKRGVSKEQIKEVLKRLGVDPKEVEEIVRRIC
Name: wt_aa_seq, Length: 312997, dtype: str

In [44]:
from src.features import get_wt_seq

X_train = get_wt_seq(X_train)
X_test = get_wt_seq(X_test)
X_train.head()

,WT_name,mut_type,aa_seq,wt_aa,mut_aa,position,protein_length,relative_position,wt_hydrophobicity,mut_hydrophobicity,...,mut_mw,delta_mw,wt_charge,mut_charge,delta_charge,wt_polarity,mut_polarity,wt_aromatic,mut_aromatic,wt_aa_seq
5,EA|run2_0325_0005.pdb,D1Q,QEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D,Q,1,47,0.021277,-3.5,-3.5,...,146.15,13.05,-1,0,1,acidic,polar,False,False,DEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
6,EA|run2_0325_0005.pdb,D1E,EEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D,E,1,47,0.021277,-3.5,-3.5,...,147.13,14.03,-1,-1,0,acidic,acidic,False,False,DEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
7,EA|run2_0325_0005.pdb,D1N,NEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D,N,1,47,0.021277,-3.5,-3.5,...,132.12,-0.98,-1,0,1,acidic,polar,False,False,DEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
8,EA|run2_0325_0005.pdb,D1H,HEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D,H,1,47,0.021277,-3.5,-3.2,...,155.15,22.05,-1,0,1,acidic,basic,False,True,DEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER
9,EA|run2_0325_0005.pdb,D1R,REVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D,R,1,47,0.021277,-3.5,-4.5,...,174.20,41.10,-1,1,2,acidic,basic,False,False,DEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER


Lets use the difference between the WT and the mut embeddings X_delta as inputs for ridge. 

In [48]:
sequences = X_train["aa_seq"].iloc[:4].tolist()

tokens = tokenizer(
    sequences,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    outputs = esm_model(**tokens)

print(outputs.last_hidden_state.shape)


torch.Size([4, 49, 320])


In [58]:
X_train['aa_seq']
positions = sequences = X_train["position"].iloc[:4].tolist()
positions

## Get 320 embeddings for mut allele
mut_embeddings = torch.stack([outputs.last_hidden_state[i, pos, :] for i, pos in enumerate(positions)])
mut_embeddings.cpu().numpy()

array([[ 0.28234148,  0.21348754,  0.01328332, ..., -0.02125118,
        -0.26223436, -0.4419395 ],
       [ 0.11875185, -0.02928586,  0.07728089, ...,  0.324715  ,
        -0.27818066, -0.14916049],
       [ 0.09355246,  0.28670368, -0.0910365 , ...,  0.3207902 ,
        -0.30418938, -0.38577   ],
       [ 0.12311812,  0.15446526,  0.04150479, ...,  0.41195825,
        -0.33333412, -0.41621566]], shape=(4, 320), dtype=float32)

In [67]:
def get_residue_embeddings(sequences, positions, tokenizer, model, batch_size=32):


    all_embeddings = []

    for start in range(0, len(sequences), batch_size):
        batch_sequences = sequences[start:start + batch_size]
        batch_positions = positions[start:start + batch_size]


        tokens = tokenizer(
            batch_sequences,
            return_tensors="pt",
            padding=True)

        with torch.no_grad():
            outputs = model(**tokens)
            
        batch_embeddings = torch.stack([
            outputs.last_hidden_state[i, pos, :]
            for i, pos in enumerate(batch_positions)])

        all_embeddings.append(batch_embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)
        

        

In [71]:



mut_embeddings = get_residue_embeddings(
    X_train["aa_seq"][1:46].tolist(),
    X_train["position"][1:46].tolist(),
    tokenizer,
    esm_model
)

mut_embeddings.shape

torch.Size([45, 320])

In [70]:
X_train["aa_seq"][1:46].tolist()

['EEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'NEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'HEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'REVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'KEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'TEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'SEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'AEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'GEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'MEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'LEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'VEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'IEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'WEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'YEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'FEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'PEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'CEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'DQVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER',
 'DNVTIHLGDK